# Tutorial 7: Train NicheTrans on 10x Xenium data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_breast_cancer import Breast_cancer

from utils.utils import *
from utils.utils_training_breast_cancer import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_breast_cancer.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.2, workers=4, adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/HBC_rep1_cell_nucleus_3channel_strength_mean.h5ad', coordinate_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/cells.csv.gz', ct_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/Cell_Barcode_Type_Matrices.xlsx', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = Breast_cancer(adata_path=args.adata_path, coordinate_path=args.coordinate_path, ct_path=args.ct_path)
trainloader, testloader = breast_cancer_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
model = nn.DataParallel(model).cuda()

------Calculating spatial graph...
The graph contains 1185564 edges, 98797 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 827796 edges, 68983 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  98797 spots, 98659 positive CD20, 84043 positive HER2 
  test     |  68983 spots, 67600 positive CD20, 36904 positive HER2 
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader)
torch.save(model.state_dict(), 'NicheTrans_breast_cancer_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40
Batch 3087/3087	 Loss 0.158334 (0.302522)
==> Epoch 2/40
Batch 3087/3087	 Loss 0.230089 (0.223546)
==> Epoch 3/40
Batch 3087/3087	 Loss 0.193684 (0.208324)
==> Epoch 4/40
Batch 3087/3087	 Loss 0.176748 (0.196835)
==> Epoch 5/40
Batch 3087/3087	 Loss 0.125602 (0.190673)
==> Epoch 6/40
Batch 3087/3087	 Loss 0.180463 (0.183386)
==> Epoch 7/40
Batch 3087/3087	 Loss 0.306314 (0.176522)
==> Epoch 8/40
Batch 3087/3087	 Loss 0.174423 (0.172967)
==> Epoch 9/40
Batch 3087/3087	 Loss 0.228722 (0.170717)
==> Epoch 10/40
Batch 3087/3087	 Loss 0.111803 (0.168846)
==> Epoch 11/40
Batch 3087/3087	 Loss 0.134777 (0.164982)
==> Epoch 12/40
Batch 3087/3087	 Loss 0.138627 (0.160666)
==> Epoch 13/40
Batch 3087/3087	 Loss 0.142668 (0.160486)
==> Epoch 14/40
Batch 3087/3087	 Loss 0.255496 (0.160513)
==> Epoch 15/40
Batch 3087/3087	 Loss 0.212001 (0.158849)
==> Epoch 16/40
Batch 3087/3087	 Loss 0.088288 (0.158560)
==> Epoch 17/40
Batch 3087/3087	 Loss 0.112012 (0.156047)
==> Epoch 18/40
Batch 3